# Getting Started with RAG using Docling

## Setup

In [1]:
!uv pip install langchain-docling langchain-core langchain-huggingface sentence-transformers langchain_milvus langchain-ibm "pymilvus[milvus_lite]" langchain-text-splitters langchain-classic langchain-openai python-dotenv

Audited 11 packages in 98ms


In [2]:
import logging
import os

from langchain_core.prompts import PromptTemplate

logging.basicConfig(level=logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

/Users/dol/codes/docling-workshops/workshops/2025_12_04/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Ingestion pipeline

In [3]:
from pathlib import Path
from tempfile import mkdtemp

from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from langchain_docling import DoclingLoader
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_milvus import Milvus

def create_retriever(*, file_path, embedding_model_id, milvus_uri=None, top_k=3):
    # create LangChain documents (chunks) with DoclingLoader
    loader = DoclingLoader(
        file_path=file_path,
        chunker=HybridChunker(
            tokenizer=HuggingFaceTokenizer.from_pretrained(
                model_name=embedding_model_id,
            ),
        ),
    )
    docs = loader.load()

    # ingest into Milvus vector store
    if milvus_uri is None:
        milvus_uri = str(Path(mkdtemp()) / "docling.db")
    vector_store = Milvus.from_documents(
        documents=docs,
        embedding=HuggingFaceEmbeddings(model_name=embedding_model_id),
        collection_name="docling_demo",
        connection_args={"uri": milvus_uri},
        index_params={"index_type": "FLAT"},
        drop_old=True,
    )
    return vector_store.as_retriever(search_kwargs={"k": top_k})


## RAG pipeline

### Choice of LLM runtime

In the Generation step of the RAG pipeline we will invoke a model. Below are a few possibilties for defining the LLM:

1. Using a local LLM engine, e.g. LM Studio, Ollama, etc. See the `get_generic_openai_api_llm()` method.
2. Using a remote LLM inference server, e.g. watsonx.ai. In this case you will might need credentials. See the `get_watsonx_llm()` method.

In [4]:
# Default parameters match to a local LM Studio instance

def get_generic_openai_api_llm(lm_model_id="ibm/granite-4-h-small", lm_base_url="http://localhost:1234/v1", lm_api_key="none"):
    from langchain_openai import ChatOpenAI    
    
    llm = ChatOpenAI(model=lm_model_id, base_url=lm_base_url, api_key=lm_api_key)
    return llm

In [5]:
def get_watsonx_llm():
    model_id = "ibm/granite-4-h-small"
    base_url = "https://us-south.ml.cloud.ibm.com"

    from langchain_ibm import ChatWatsonx
    from dotenv import load_dotenv

    load_dotenv()
    api_key = os.environ.get("WX_API_KEY")
    project_id = os.environ.get("WX_PROJECT_ID")
    if api_key is None or project_id is None:
        raise RuntimeError("An API key for watsonx is required to run this part of the notebook. Please set WX_API_KEY and WX_PROJECT_ID in the .env file.")

    generation_params = {
        "temperature": 0.7,           # 0.0 (deterministic) to 1.0 (creative)
        "max_tokens": 1000,           # Maximum output length
        "top_p": 0.9,                 # Nucleus sampling threshold
    }
    llm = ChatWatsonx(
            model_id=model_id,
            url=base_url,
            project_id=project_id,
            apikey=api_key,
            params=generation_params  # Pass the structured params
        )
    return llm


### Defining the RAG pipeline

In [6]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

def clip_text(text, limit=100):
    return f"{text[:limit]}..." if len(text) > limit else text

def do_rag(*, retriever, question, llm, lm_prompt):
    question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=lm_prompt)
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)
    resp_dict = rag_chain.invoke({"input": question})

    print(f"Question:\n{resp_dict['input']}\n\nAnswer:\n{clip_text(resp_dict['answer'], limit=1000)}")
    return resp_dict

def print_sources(resp_dict):
    for i, doc in enumerate(resp_dict["context"]):
        print(f"Source {i+1}:")
        print(f"  text: {clip_text(doc.page_content)}")
        for key in doc.metadata:
            if key != "pk":
                val = doc.metadata.get(key)
                clipped_val = clip_text(val, limit=100) if isinstance(val, str) else val
                print(f"  {key}: {clipped_val}")

## Running RAG end-to-end

Running an end-to-end example in English:


In [7]:
retriever = create_retriever(
    file_path="https://arxiv.org/pdf/2408.09869",
    embedding_model_id="ibm-granite/granite-embedding-30m-english",
)

Token indices sequence length is longer than the specified maximum sequence length for this model (619 > 512). Running this sequence through the model will result in indexing errors
/Users/dol/codes/docling-workshops/workshops/2025_12_04/.venv/lib/python3.12/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


<div class="alert alert-info">
    <strong>INFO</strong>: Using the <code>HybridChunker</code> can sometimes lead to a warning from the transformers library, however this is a "false alarm" — for details check <a href="https://docling-project.github.io/docling/faq/#hybridchunker-triggers-warning-token-indices-sequence-length-is-longer-than-the-specified-maximum-sequence-length-for-this-model">here</a>.
</div>

In [8]:
rag_result = do_rag(
    retriever=retriever,
    question="Briefly name the main AI models used in Docling.",

    # using a local model
    # llm=get_generic_openai_api_llm(),

    # using watsonx.ai
    llm=get_watsonx_llm(),

    lm_prompt=PromptTemplate.from_template("Context information is below.\n---------------------\n{context}\n---------------------\nGiven the context information and not prior knowledge, answer the query.\nQuery: {input}\nAnswer:\n"),
)

Question:
Briefly name the main AI models used in Docling.

Answer:
Based on the provided context, the two main AI models used in Docling are:

1. Layout analysis model - an accurate object-detector for page elements.

2. TableFormer - a state-of-the-art table structure recognition model.


In [9]:
print_sources(rag_result)

Source 1:
  text: 3.2 AI models
As part of Docling, we initially release two highly capable AI models to the open-sour...
  source: https://arxiv.org/pdf/2408.09869
  dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/50', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 3, 'bbox': {'l': 108.0, 't': 404.873, 'r': 504.003, 'b': 330.866, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 608]}]}], 'headings': ['3.2 AI models'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 11465328351749295394, 'filename': '2408.09869v5.pdf'}}
Source 2:
  text: 6 Future work and contributions
Docling is designed to allow easy extension of the model library and...
  source: https://arxiv.org/pdf/2408.09869
  dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/76', 'parent': {'$ref': '#/body'}, 'ch